# 03 Activations and Metadata

This notebook audits saved activations and per-operation metadata. A PyTorch user usually starts with shapes and tensors, then moves to dtype, memory, arguments, RNG state, and source-code context when debugging.

The model below uses dropout so RNG capture has something meaningful to record. We set the model to train mode intentionally and fix `random_seed` for reproducibility.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import torch
from torch import nn
import torchlens as tl

torch.manual_seed(17)


class MetadataNet(nn.Module):
    """Small model for activation and metadata inspection."""

    def __init__(self) -> None:
        """Initialize layers."""
        super().__init__()
        self.fc1 = nn.Linear(4, 4)
        self.drop = nn.Dropout(p=0.25)
        self.fc2 = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor, scale: float = 1.0) -> torch.Tensor:
        """Run a forward pass with a non-tensor argument."""
        h = torch.relu(self.fc1(x))
        h = self.drop(h) * scale
        return self.fc2(h)


model = MetadataNet().train()
x = torch.randn(3, 4)
trace = tl.trace(
    model,
    (x,),
    input_kwargs={"scale": 1.5},
    save_arg_values=True,
    save_code_context=True,
    save_rng_states=True,
    random_seed=123,
)
print(trace.summary(fields=["name", "shape", "params"]))

Saved activations are exposed as tensors through `out`/`tensor`. The layer still carries metadata such as shape and dtype even when you only need lightweight inspection.

In [ ]:
for layer in trace.layers:
    tensor = getattr(layer, "out", None)
    tensor_shape = tuple(tensor.shape) if isinstance(tensor, torch.Tensor) else None
    print(
        f"{layer.layer_label:14s} shape={str(layer.shape):12s} "
        f"dtype={str(layer.dtype):14s} tensor_shape={tensor_shape}"
    )

> NOTE: The v9 glossary names the output memory field `activation_memory`, but the current executable code exposes `memory` and `memory_str` on `Op`/`Layer`, plus aggregate fields such as `total_activation_memory_str` on `Trace`. This notebook uses the names that run.

In [ ]:
print(f"trace total activation memory: {trace.total_activation_memory_str}")
print(f"trace saved activation memory: {trace.saved_activation_memory_str}")
for layer in trace.layers:
    print(f"{layer.layer_label:14s} memory={layer.memory_str}")

sample_memory = trace[1].memory
print(f"\nRaw memory value type: {type(sample_memory).__name__}; value={sample_memory}")
print(
    f"Top-level Quantity classes exported: {[name for name in ['Quantity', 'Bytes', 'Duration', 'Flops', 'Macs'] if hasattr(tl, name)]}"
)

Argument summaries and saved argument values help explain why an op produced a given tensor. Some boundary records do not have argument metadata, so this cell filters to ops that do.

In [ ]:
printed = 0
for op in trace.ops:
    args_summary = getattr(op, "args_summary", None)
    kwargs_summary = getattr(op, "kwargs_summary", None)
    saved_args = getattr(op, "args", None)
    saved_kwargs = getattr(op, "kwargs", None)
    if any(value is not None for value in [args_summary, kwargs_summary, saved_args, saved_kwargs]):
        print(
            f"{op.layer_label:14s} args_summary={args_summary} kwargs_summary={kwargs_summary} "
            f"saved_args={type(saved_args).__name__ if saved_args is not None else None} "
            f"saved_kwargs={type(saved_kwargs).__name__ if saved_kwargs is not None else None}"
        )
        printed += 1
if printed == 0:
    print("no args_summary or saved argument values are populated for these ops in this build")

RNG capture is useful for stochastic models. Current records expose function-level RNG state collections through `func_rng_states` on ops/layers.

In [ ]:
for op in trace.ops:
    rng_states = getattr(op, "func_rng_states", None)
    if rng_states:
        print(f"{op.layer_label:14s} captured RNG entries: {len(rng_states)}")
        first_key = next(iter(rng_states))
        print(f"  first key: {first_key!r}")
        print(f"  first entry type: {type(rng_states[first_key]).__name__}")
        break
else:
    print("No per-op RNG entries were captured in this run.")

Code context points from a captured operation back to the Python stack. For notebook-defined code, the exact source text may vary by environment, but file/function/line identity should be visible when captured.

In [ ]:
for op in trace.ops:
    if op.code_context:
        frame = op.code_context[0]
        print(f"op: {op.layer_label}")
        print(f"file: {frame.file}")
        print(f"function: {frame.func_name}")
        print(f"line: {frame.line_number}")
        print(f"source available: {bool(getattr(frame, 'source_context', None))}")
        break
else:
    print("No code context was captured for surviving ops.")

Trace-level timing currently appears as capture start/end timestamps. This is enough to audit capture duration without relying on per-op timing fields that are not exposed in this build.

In [ ]:
elapsed = trace.capture_end_time - trace.capture_start_time
print(f"capture_start_time: {trace.capture_start_time:.6f}")
print(f"capture_end_time:   {trace.capture_end_time:.6f}")
print(f"elapsed seconds:    {elapsed:.6f}")

## 🔧 Sandbox

Mini-experiment: change `sandbox_dropout_p`, `sandbox_scale`, and `sandbox_seed`. Expected observation: the dropout/multiply activations move, while shapes stay fixed. The cell prints baseline vs sandbox sums.

In [ ]:
sandbox_dropout_p = 0.75
sandbox_scale = 2.0
sandbox_seed = 321

baseline_output = trace[trace.output_layers[0]].out.detach()
model.drop.p = sandbox_dropout_p
sandbox_trace = tl.trace(
    model,
    (x,),
    input_kwargs={"scale": sandbox_scale},
    save_arg_values=True,
    save_rng_states=True,
    random_seed=sandbox_seed,
)
sandbox_output = sandbox_trace[sandbox_trace.output_layers[0]].out.detach()
print(f"dropout p: 0.25 -> {sandbox_dropout_p}")
print(f"scale: 1.5 -> {sandbox_scale}")
print(f"output sum: {baseline_output.sum().item():.6f} -> {sandbox_output.sum().item():.6f}")
print(f"max output delta: {(sandbox_output - baseline_output).abs().max().item():.6f}")